1. EDA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", 50)

In [ ]:
df = pd.read_csv("data.csv")
df.head()

In [ ]:
# В датасете есть пустой столбец 'Unnamed: 32' и id — они не нужны
df = df.drop(columns=[c for c in df.columns if "Unnamed" in c or c == "id"])
df.info()

In [ ]:
df.describe()
df["diagnosis"].value_counts()

In [ ]:
features = df.columns.drop("diagnosis")

fig, axes = plt.subplots(6, 5, figsize=(20, 20))
for ax, col in zip(axes.ravel(), features):
    sns.histplot(data=df, x=col, hue="diagnosis",
                 kde=True, ax=ax, bins=25, alpha=0.5)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(18, 14))
corr = df.select_dtypes("number").corr()
sns.heatmap(corr, cmap="coolwarm", center=0, annot=False, square=True)
plt.title("Матрица корреляций")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(data=df, x="radius_mean", y="perimeter_mean",
                hue="diagnosis", ax=axes[0])
sns.scatterplot(data=df, x="radius_mean", y="area_mean",
                hue="diagnosis", ax=axes[1])
sns.scatterplot(data=df, x="perimeter_mean", y="area_mean",
                hue="diagnosis", ax=axes[2])

plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(6, 5, figsize=(20, 20))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(data=df, x="diagnosis", y=col, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

2. Моделирование kNN

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_curve, roc_auc_score)

X = df.drop(columns="diagnosis")
y = (df["diagnosis"] == "M").astype(int)   # M=1, B=0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
def evaluate(model, X_tr, y_tr, X_te, y_te, name):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        "model": name,
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred),
        "recall": recall_score(y_te, pred),
        "f1": f1_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
    }, proba

knn_default = KNeighborsClassifier()
metrics_knn, proba_knn = evaluate(knn_default, X_train_s, y_train, X_test_s, y_test, "kNN (k=5)")
print(metrics_knn)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, proba_knn)
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label=f"kNN (AUC = {metrics_knn['roc_auc']:.3f})")
plt.plot([0, 1], [0, 1], "--", color="grey")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC-кривая kNN")
plt.legend()
plt.show()

In [ ]:
from sklearn.model_selection import cross_val_score

ks = range(1, 31)
cv_scores = [cross_val_score(KNeighborsClassifier(n_neighbors=k),
                              X_train_s, y_train, cv=5, scoring="roc_auc").mean()
             for k in ks]

best_k = ks[int(np.argmax(cv_scores))]
print(f"Лучшее k = {best_k}, CV ROC-AUC = {max(cv_scores):.3f}")

plt.figure(figsize=(10, 5))
plt.plot(ks, cv_scores, marker="o")
plt.axvline(best_k, color="red", linestyle="--", label=f"best k = {best_k}")
plt.xlabel("k"); plt.ylabel("CV ROC-AUC")
plt.title("Зависимость качества kNN от k")
plt.legend(); plt.grid(True); plt.show()

In [ ]:
knn_tuned = KNeighborsClassifier(n_neighbors=best_k)
metrics_knn_tuned, proba_knn_tuned = evaluate(
    knn_tuned, X_train_s, y_train, X_test_s, y_test, f"kNN (k={best_k})"
)

pd.DataFrame([metrics_knn, metrics_knn_tuned]).round(4)

In [ ]:
corr_matrix = X_train.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.85)]
print("Удаляем признаки:", to_drop)

X_train_lr = X_train.drop(columns=to_drop)
X_test_lr  = X_test.drop(columns=to_drop)

scaler_lr = StandardScaler()
X_train_lr_s = scaler_lr.fit_transform(X_train_lr)
X_test_lr_s  = scaler_lr.transform(X_test_lr)

In [ ]:
from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=5000)
metrics_lr, proba_lr = evaluate(logreg, X_train_lr_s, y_train, X_test_lr_s, y_test, "LogReg")
print(metrics_lr)

In [ ]:
coefs = pd.Series(logreg.coef_[0], index=X_train_lr.columns).sort_values()

plt.figure(figsize=(10, 8))
coefs.plot(kind="barh")
plt.title("Коэффициенты логистической регрессии")
plt.grid(True, axis="x")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.linear_model import LogisticRegressionCV

logreg_cv = LogisticRegressionCV(
    Cs=np.logspace(-3, 3, 20), cv=5, scoring="roc_auc",
    max_iter=5000, random_state=42
)
metrics_lr_cv, proba_lr_cv = evaluate(
    logreg_cv, X_train_lr_s, y_train, X_test_lr_s, y_test, "LogRegCV"
)
print(metrics_lr_cv)

# Визуализация коэффициентов после настройки
coefs_cv = pd.Series(logreg_cv.coef_[0], index=X_train_lr.columns).sort_values()
plt.figure(figsize=(10, 8))
coefs_cv.plot(kind="barh")
plt.title(f"Коэффициенты LogRegCV (C = {logreg_cv.C_[0]:.3f})")
plt.grid(True, axis="x")
plt.tight_layout(); plt.show()

In [ ]:
comparison = pd.DataFrame([
    metrics_knn, metrics_knn_tuned, metrics_lr, metrics_lr_cv
]).round(4)
comparison

На этом датасете обе модели очень близки (ROC-AUC ~0.98–0.99). kNN даёт чуть более высокий recall (важно для медицины — пропустить злокачественную опухоль хуже, чем ошибиться с доброкачественной), логистическая регрессия — лучше интерпретируется (можно объяснить, какие признаки и как влияют). Если приоритет — не пропустить рак, выбираем kNN или LogReg с акцентом на recall. Если приоритет — интерпретируемость врачу, выбираем логистическую регрессию.